In [1]:
#!pip install -q transformers accelerate #1.5b model requirements
# Cell 1: Install
!pip install -q transformers accelerate bitsandbytes pylint bandit #7b requiremnt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.7/134.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.4/276.4 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 3.0 MB/s eta 0:00:00


In [2]:
# Cell 2: Load the 7B model
import torch
from transformers import pipeline, BitsAndBytesConfig

model = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-Coder-7B-Instruct",
    device_map="auto",
    model_kwargs={"quantization_config": BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
    )},
)
print("7B model loaded!")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

7B model loaded!


In [3]:
#Cell 3: Prompt
system_prompt = """You are a strict Python code reviewer. Check every line for:
- bug: logic errors, wrong comparisons (> vs >=), crashes on empty input,
  division by zero, off-by-one errors, mutable default arguments
- security: eval, exec, shell commands, passwords in code
- style: unused imports, names that break PEP 8 (only if they really break it)
- optimization: slow or unnecessary code

Think about edge cases: empty lists, zero, None, boundary values.

Reply ONLY with a JSON list. Each item has: "line", "category", "message", "fix", "evidence".
"evidence" must be the exact code copied from that line.

Example input:
1: import sys
2: def first_item(values):
3:     return values[0]
4: def run(a):
5:     return exec(a)

Example output:
[{"line": 1, "category": "style", "message": "sys is imported but never used", "fix": "remove the import", "evidence": "import sys"},
 {"line": 3, "category": "bug", "message": "crashes with IndexError when values is empty", "fix": "check if values is empty first", "evidence": "return values[0]"},
 {"line": 5, "category": "security", "message": "exec runs any code the user gives", "fix": "avoid exec; parse input safely", "evidence": "return exec(a)"}]"""

In [4]:
#Cell 4: AI review function
import json

def clean_json(text):
    start = text.find("[")
    end = text.rfind("]")
    if start == -1 or end == -1:
        return []
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return []


def llm_review(code):
    numbered = ""
    for i, line in enumerate(code.split("\n"), start=1):
        numbered += f"{i}: {line}\n"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "Review this code:\n" + numbered},
    ]
    output = model(messages, max_new_tokens=500, do_sample=False)
    answer = output[0]["generated_text"][-1]["content"]

    found = []
    for item in clean_json(answer):
        found.append({
            "line": item.get("line", 0),
            "category": item.get("category", "bug") if item.get("category") in ["bug", "security", "style", "optimization"] else "bug",
            "message": item.get("message", ""),
            "fix": item.get("fix", ""),
            "evidence": item.get("evidence", ""),
            "tool": "llm",
        })
    return found

In [7]:
#Cell 5: Combine and check
from analyzer import run_pylint, run_bandit


def is_real(issue, code_lines):
    line_no = issue["line"]
    if line_no < 1 or line_no > len(code_lines):
        return False

    evidence = issue["evidence"].replace(" ", "")
    if evidence == "":
        return True

    for n in [line_no - 1, line_no, line_no + 1]:
        if 1 <= n <= len(code_lines):
             if evidence in code_lines[n - 1].replace(" ", ""):
                issue["line"] = n          # correct the AI's line number
                return True
    return False


def review(code):
    code_lines = code.split("\n")

    with open("sample.py", "w") as f:
        f.write(code)
    static_issues = run_pylint("sample.py") + run_bandit("sample.py")

    ai_issues = []
    removed = 0
    for issue in llm_review(code):
        if is_real(issue, code_lines):
            ai_issues.append(issue)
        else:
            removed += 1
    print(f"Removed {removed} fake AI answer(s)")

    final = {}
    for issue in static_issues + ai_issues:
        key = (issue["line"], issue["category"])
        if key in final:
            if issue["tool"] not in final[key]["tools"]:
                final[key]["tools"].append(issue["tool"])
            if issue["tool"] != "llm" and issue["message"] not in final[key]["message"]:
                final[key]["message"] += " / " + issue["message"]
            if issue.get("fix"):
                final[key]["fix"] = issue["fix"]
        else:
            issue["tools"] = [issue["tool"]]
            final[key] = issue

    return sorted(final.values(), key=lambda x: x["line"])

In [8]:
#Cell 6: Test with the bad code
code = """import os

def add(x, items=[]):
    items.append(x)
    return eval(input())
"""

for issue in review(code):
    print(issue["line"], issue["category"], issue["tools"], issue["message"])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Removed 0 fake AI answer(s)
1 style ['pylint'] Unused import os
3 bug ['pylint', 'llm'] Dangerous default value [] as argument
5 security ['pylint', 'bandit'] Use of eval / Use of possibly insecure function - consider using safer ast.literal_eval.
5 bug ['llm'] eval can execute arbitrary code


In [9]:

#Cell 7: Test the fake-answer check
code_lines = code.split("\n")

fake = {"line": 3, "category": "security", "message": "password hard-coded", "evidence": "password = '123'", "tool": "llm"}
real = {"line": 5, "category": "security", "message": "eval is unsafe", "evidence": "eval(input())", "tool": "llm"}

print("Fake answer kept?", is_real(fake, code_lines))
print("Real answer kept?", is_real(real, code_lines))

Fake answer kept? False
Real answer kept? True


In [10]:
#Cell 8: Test logic bugs
logic_code = """def is_adult(age):
    return age > 18


def average(numbers):
    return sum(numbers) / len(numbers)
"""

for issue in review(logic_code):
    print(issue["line"], issue["category"], issue["tools"], issue["message"])

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Removed 0 fake AI answer(s)
6 bug ['llm'] division by zero when numbers is empty


In [11]:
#New cell (Cell 9):
!pip install -q gradio

In [12]:
#cELL 10:The Gradio web page
import gradio as gr


def review_ui(code_text):
    issues = review(code_text)
    if not issues:
        return "No issues found."

    text = ""
    for issue in issues:
        text += f"Line {issue['line']} | {issue['category'].upper()} | found by: {', '.join(issue['tools'])}\n"
        text += f"   Problem: {issue['message']}\n"
        if issue.get("fix"):
            text += f"   Fix: {issue['fix']}\n"
        text += "\n"
    return text


demo = gr.Interface(
    fn=review_ui,
    inputs=gr.Code(language="python", label="Paste your Python code"),
    outputs=gr.Textbox(label="Review result", lines=20),
    title="Code Review Assistant",
    description="Static analysis (pylint + bandit) combined with an open-source AI model (Qwen 7B).",
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://473196c83158eb8625.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [13]:
# Cell 11: Test cases (code with known answers for evaluation)

test_cases = [
    {
        "name": "mutable default",
        "code": """def add_tag(tag, tags=[]):
    tags.append(tag)
    return tags
""",
        "expected": [(1, "bug")],
    },
    {
        "name": "shell injection",
        "code": """import subprocess


def ping(host):
    return subprocess.call("ping " + host, shell=True)
""",
        "expected": [(5, "security")],
    },
    {
        "name": "clean code",
        "code": """def square(x):
    return x * x
""",
        "expected": [],
    },
    {
        "name": "empty average",
        "code": """def average(values):
    return sum(values) / len(values)
""",
        "expected": [(2, "bug")],
    },
    {
        "name": "eval input",
        "code": """def calculate(text):
    return eval(text)
""",
        "expected": [(2, "security")],
    },
    {
        "name": "hardcoded password",
        "code": """PASSWORD = "admin123"


def login(user):
    return user == "admin"
""",
        "expected": [(1, "security")],
    },
    {
        "name": "unused import",
        "code": """import sys


def greet(name):
    return "Hello " + name
""",
        "expected": [(1, "style")],
    },
    {
        "name": "bad function name",
        "code": """def CalculateTotal(a, b):
    return a + b
""",
        "expected": [(1, "style")],
    },
    {
        "name": "none comparison",
        "code": """def has_value(x):
    if x == None:
        return False
    return True
""",
        "expected": [(2, "style")],
    },
    {
        "name": "slow string",
        "code": """def join_words(words):
    result = ""
    for word in words:
        result += word
    return result
""",
        "expected": [(4, "optimization")],
    },
    {
        "name": "first item crash",
        "code": """def first(items):
    return items[0]
""",
        "expected": [(2, "bug")],
    },
    {
        "name": "clean code 2",
        "code": """def is_even(n):
    return n % 2 == 0
""",
        "expected": [],
    },
]

print(len(test_cases), "test cases ready")

12 test cases ready


In [14]:
for case in test_cases:
    print("=====", case["name"], "| expected:", case["expected"])
    for i, line in enumerate(case["code"].split("\n"), start=1):
        print(f"{i}: {line}")
    print()

===== mutable default | expected: [(1, 'bug')]
1: def add_tag(tag, tags=[]):
2:     tags.append(tag)
3:     return tags
4: 

===== shell injection | expected: [(5, 'security')]
1: import subprocess
2: 
3: 
4: def ping(host):
5:     return subprocess.call("ping " + host, shell=True)
6: 

===== clean code | expected: []
1: def square(x):
2:     return x * x
3: 

===== empty average | expected: [(2, 'bug')]
1: def average(values):
2:     return sum(values) / len(values)
3: 

===== eval input | expected: [(2, 'security')]
1: def calculate(text):
2:     return eval(text)
3: 

===== hardcoded password | expected: [(1, 'security')]
1: PASSWORD = "admin123"
2: 
3: 
4: def login(user):
5:     return user == "admin"
6: 

===== unused import | expected: [(1, 'style')]
1: import sys
2: 
3: 
4: def greet(name):
5:     return "Hello " + name
6: 

===== bad function name | expected: [(1, 'style')]
1: def CalculateTotal(a, b):
2:     return a + b
3: 

===== none comparison | expected: [(2, 'style')]
1

In [15]:
# Cell 12: Evaluation - precision, recall, F1 for static vs llm vs hybrid

import time
import importlib
import analyzer
importlib.reload(analyzer)                 # load the updated analyzer.py
from analyzer import run_pylint, run_bandit


def get_predictions(code, mode):
    if mode == "static":
        with open("sample.py", "w") as f:
            f.write(code)
        issues = run_pylint("sample.py") + run_bandit("sample.py")
    elif mode == "llm":
        code_lines = code.split("\n")
        issues = [i for i in llm_review(code) if is_real(i, code_lines)]
    else:
        issues = review(code)
    # keep only (line, category), with no duplicates
    return list({(i["line"], i["category"]) for i in issues})


def count_matches(predicted, expected):
    tp = 0
    remaining = list(expected)
    for p_line, p_cat in predicted:
        for e in remaining:
            if p_cat == e[1] and abs(p_line - e[0]) <= 1:   # same category, line within 1
                tp += 1
                remaining.remove(e)
                break
    fp = len(predicted) - tp
    fn = len(remaining)
    return tp, fp, fn


results = {}
for mode in ["static", "llm", "hybrid"]:
    total_tp = total_fp = total_fn = 0
    start = time.time()
    for case in test_cases:
        predicted = get_predictions(case["code"], mode)
        tp, fp, fn = count_matches(predicted, case["expected"])
        total_tp += tp
        total_fp += fp
        total_fn += fn
    seconds = time.time() - start

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    results[mode] = (precision, recall, f1, total_tp, total_fp, total_fn, seconds)

print(f"{'Mode':<8} {'Precision':>9} {'Recall':>7} {'F1':>5} {'TP':>4} {'FP':>4} {'FN':>4} {'Time(s)':>8}")
for mode, (p, r, f1, tp, fp, fn, s) in results.items():
    print(f"{mode:<8} {p:>9.2f} {r:>7.2f} {f1:>5.2f} {tp:>4} {fp:>4} {fn:>4} {s:>8.1f}")

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Removed 0 fake AI answer(s)


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Removed 0 fake AI answer(s)


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Removed 1 fake AI answer(s)


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Removed 0 fake AI answer(s)


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Removed 0 fake AI answer(s)
Mode     Precision  Recall    F1   TP   FP   FN  Time(s)
static        0.89    0.80  0.84    8    1    2     29.8
llm           0.73    0.80  0.76    8    3    2     78.3
hybrid        0.71    1.00  0.83   10    4    0     85.7


Cell 1   install
Cell 2   load model
Cell 3   prompt
Cell 4   llm_review
Cell 5   review + is_real
Cell 6   test bad code
Cell 7   test fake answer
Cell 8   test logic bugs
Cell 9   install gradio
Cell 10  Gradio web page
Cell 11  test_cases
         helper (prints test cases)
Cell 12  evaluation   ← here

In [16]:
# Cell 13: Error analysis - what did hybrid get wrong?

for case in test_cases:
    predicted = get_predictions(case["code"], "hybrid")
    expected = case["expected"]
    extra = [p for p in predicted if not any(p[1] == e[1] and abs(p[0] - e[0]) <= 1 for e in expected)]
    missed = [e for e in expected if not any(p[1] == e[1] and abs(p[0] - e[0]) <= 1 for p in predicted)]
    if extra or missed:
        print(case["name"], "| extra:", extra, "| missed:", missed)

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Removed 0 fake AI answer(s)


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Removed 0 fake AI answer(s)
shell injection | extra: [(1, 'security')] | missed: []


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Removed 1 fake AI answer(s)


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Removed 0 fake AI answer(s)


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Removed 0 fake AI answer(s)
clean code 2 | extra: [(1, 'style')] | missed: []
